# PySpark — Null Handling

Real-world data always has missing values. PySpark represents missing values as `null` (not Python's `None` in the DataFrame context).

| Operation | Method | Purpose |
|-----------|--------|---------|
| Check for null | `isNull()` / `isNotNull()` | Filter rows by null status |
| Drop null rows | `dropna()` / `df.na.drop()` | Remove rows with any/some nulls |
| Fill null values | `fillna()` / `df.na.fill()` | Replace nulls with a default value |
| Replace values | `df.na.replace()` | Swap specific values |

**Key rule:** `null != null` in SQL/Spark — always use `isNull()` / `isNotNull()`, never `== null`.

## Setup — DataFrame with Nulls

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, isnan, when, count, coalesce, lit

active = SparkSession.getActiveSession()
if active:
    active.stop()

spark = SparkSession.builder \
    .appName("NullHandling") \
    .config("spark.pyspark.python", sys.executable) \
    .getOrCreate()

data = [
    (1, "Alice",   "Engineering", 95000, 4.5),
    (2, "Bob",     None,          72000, None),
    (3, "Charlie", "Engineering", None,  5.0),
    (4, "Diana",   "HR",          65000, 3.8),
    (5, "Eve",     None,          None,  None),
    (6, "Frank",   "Marketing",   85000, 4.1),
]
columns = ["id", "name", "dept", "salary", "rating"]
df = spark.createDataFrame(data, columns)
df.show()
df.printSchema()

## 1. isNull() and isNotNull() — Check for Nulls

**Never use `== None` or `== null`** — in Spark, `null == null` is `null`, not `True`.

In [ ]:
# Filter rows where dept is null
df.filter(col("dept").isNull()).show()

# Filter rows where dept is NOT null
df.filter(col("dept").isNotNull()).show()

# Count nulls in each column
print("Null counts per column:")
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

# This is WRONG — never do this
# df.filter(col("dept") == None)   # always returns 0 rows

## 2. dropna() — Drop Rows with Nulls

`dropna(how, thresh, subset)` — remove rows based on null presence.

In [ ]:
# Drop rows where ANY column is null
df.dropna().show()
print("Original rows:", df.count(), "→ After dropna():", df.dropna().count())

print()

# Drop rows where ALL columns are null
df.dropna(how="all").show()

# Drop rows where specific columns have nulls
df.dropna(subset=["dept", "salary"]).show()

# Keep rows with at least N non-null values (thresh)
# thresh=4 means keep row if at least 4 columns are non-null
df.dropna(thresh=4).show()

## 3. fillna() — Replace Nulls with Default Values

`fillna(value, subset)` — replace nulls with a constant value or a dict of column-specific values.

In [ ]:
# Fill all string columns with "Unknown"
df.fillna("Unknown").show()

# Fill all numeric columns with 0
df.fillna(0).show()

# Fill specific columns with different values — use a dict
df.fillna({
    "dept":   "Unassigned",
    "salary":  0,
    "rating":  0.0
}).show()

# Fill only specific columns
df.fillna("Unknown", subset=["dept"]).show()

## 4. coalesce() — First Non-Null Value

`coalesce(col1, col2, ...)` returns the **first non-null value** from the list of columns — very useful for fallback logic.

In [ ]:
# If dept is null, fall back to "Unknown"
df.withColumn("dept_safe", coalesce(col("dept"), lit("Unknown"))).show()

# Multiple fallbacks
df_priority = spark.createDataFrame([
    (1, "Engineering", None, None),
    (2, None, "Tech", None),
    (3, None, None, "IT"),
    (4, None, None, None),
], ["id", "dept_primary", "dept_secondary", "dept_fallback"])

df_priority.withColumn(
    "dept",
    coalesce(col("dept_primary"), col("dept_secondary"), col("dept_fallback"), lit("Unknown"))
).show()

## 5. when() / otherwise() — Conditional Null Replacement

For more complex logic than a simple fill — like filling with computed values.

In [ ]:
# Replace null salary with 60000 (default), flag the record
df_clean = df.withColumn(
    "salary_clean",
    when(col("salary").isNull(), 60000).otherwise(col("salary"))
).withColumn(
    "data_quality",
    when(col("dept").isNull() | col("salary").isNull(), "INCOMPLETE").otherwise("OK")
)
df_clean.show()

## 6. na.replace() — Replace Specific Values

`df.na.replace()` replaces specific values (can also replace a non-null value with null).

In [ ]:
# Replace specific string values
df.na.replace({"Engineering": "ENG", "Marketing": "MKT", "HR": "HRM"}, subset=["dept"]).show()

# Replace a number value
df.na.replace(72000, 75000, subset=["salary"]).show()

## 7. Real-World: Data Quality Pipeline

In [ ]:
# Full pipeline: audit → clean → report
from pyspark.sql.functions import count, when, col

# Step 1: Audit — count nulls per column
print("=== NULL AUDIT ===")
null_counts = df.select([
    count(when(col(c).isNull(), 1)).alias(c) for c in df.columns
])
null_counts.show()

# Step 2: Clean
df_cleaned = (
    df
    .fillna({"dept": "Unassigned", "salary": 0, "rating": 0.0})
    .withColumn("is_complete", ~(col("dept") == "Unassigned") & (col("salary") > 0))
)

print("=== CLEANED DATA ===")
df_cleaned.show()

# Step 3: Report
complete = df_cleaned.filter(col("is_complete")).count()
total    = df_cleaned.count()
print(f"Complete records: {complete}/{total} ({complete/total*100:.0f}%)")

## Quick Reference

```python
from pyspark.sql.functions import col, when, coalesce, lit, count

# Check for null
col("x").isNull()          # True if null
col("x").isNotNull()       # True if not null

# Filter rows
df.filter(col("x").isNull())      # rows where x is null
df.filter(col("x").isNotNull())   # rows where x is not null

# Drop rows
df.dropna()                            # any null → drop row
df.dropna(how="all")                   # only if ALL are null
df.dropna(subset=["col1", "col2"])     # nulls in specific columns
df.dropna(thresh=3)                    # keep if at least 3 non-null

# Fill nulls
df.fillna(0)                           # all numeric columns
df.fillna("Unknown")                   # all string columns
df.fillna({"col1": 0, "col2": "N/A"}) # per column

# Coalesce — first non-null
df.withColumn("safe", coalesce(col("x"), lit("default")))

# Conditional fill
df.withColumn("x", when(col("x").isNull(), 99).otherwise(col("x")))

# Replace specific values
df.na.replace({"old": "new"}, subset=["col"])

# Count nulls per column
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
```

## Interview Questions

1. Why can't you use `== None` to check for nulls in PySpark?
2. What is the difference between `dropna(how='any')` and `dropna(how='all')`?
3. What does `fillna()` do when you pass a dict?
4. What is `coalesce()` and when would you prefer it over `fillna()`?
5. How do you count the number of nulls in each column?
6. What does `thresh` do in `dropna()`?
7. What is the difference between `fillna()` and `na.replace()`?

## Practice Exercises

**Easy:**
1. Filter `df` to show only rows where `salary` is null.
2. Replace null `dept` with `"Unknown"` and null `salary` with `0`.

**Medium:**
3. Count nulls in each column — produce a null audit report.
4. Drop rows where both `dept` AND `salary` are null.
5. Add a `data_quality` column: `"COMPLETE"` if no nulls in any column, else `"INCOMPLETE"`.

**Advanced:**
6. Write a function `null_audit(df)` that returns a DataFrame summarising null counts and percentages for each column.
7. Create a pipeline that fills nulls with column medians (for numeric) and mode (for string).

In [ ]:
spark.stop()